In [ ]:
# Imports and global configuration
import os
import glob
import math
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display
import soundfile as sf

from tqdm import tqdm
from joblib import dump, load
from concurrent.futures import ProcessPoolExecutor, as_completed

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

import joblib
import warnings
warnings.filterwarnings("ignore")

# Paths (edit these paths to match your local setup)
DATASET_PATH = r"C:\Users\user\Classically\classically_punk_music_genres"     # path to extracted dataset folder
CACHE_FILE = "features_cache.joblib"
MODEL_FILE = "genre_classifier.joblib"
REG_MODEL_FILE = "spectral_centroid_regressor.joblib"
PPTX_FILE = "genre_classification_report.pptx"

RANDOM_STATE = 42
SAMPLE_RATE = 22050  # librosa default
DURATION = None  # load entire file


In [ ]:
# Build list of audio files and labels
def find_audio_files(dataset_path):
    dataset_path = Path(dataset_path)
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset path {dataset_path} does not exist. Please extract dataset locally.")
    audio_extensions = ("*.wav", "*.mp3", "*.au", "*.aif", "*.aiff", "*.flac")
    records = []
    # assume each subfolder is a genre label
    for genre_dir in sorted([p for p in dataset_path.iterdir() if p.is_dir()]):
        genre = genre_dir.name
        for ext in audio_extensions:
            for file in genre_dir.rglob(ext):
                records.append({"path": str(file), "genre": genre})
    df_files = pd.DataFrame(records)
    print(f"Found {len(df_files)} files across {df_files['genre'].nunique()} genres.")
    return df_files

df_files = find_audio_files(DATASET_PATH)
df_files.head()


In [ ]:
# Feature extraction for a single file
def extract_features(file_path, sr=SAMPLE_RATE, duration=DURATION):
    try:
        y, sr = librosa.load(file_path, sr=sr, mono=True, duration=duration)
        # ensure non-empty
        if y.size == 0:
            return None
        # Basic stats
        rms = np.mean(librosa.feature.rms(y=y))
        # Tempo (bpm)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        # Spectral features
        spec_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        spec_bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
        spec_contrast = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr))
        spec_rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        # MFCCs
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_means = np.mean(mfcc, axis=1)
        # Chroma
        chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr), axis=1)
        # Tonnetz (needs harmonic)
        y_harmonic = librosa.effects.harmonic(y)
        try:
            tonnetz = np.mean(librosa.feature.tonnetz(y=y_harmonic, sr=sr), axis=1)
        except Exception:
            tonnetz = np.zeros(6)
        features = {
            "rms": float(rms),
            "tempo": float(tempo),
            "spec_centroid": float(spec_centroid),
            "spec_bandwidth": float(spec_bandwidth),
            "spec_contrast": float(spec_contrast),
            "spec_rolloff": float(spec_rolloff),
            "zcr": float(zcr)
        }
        # add mfcc and chroma and tonnetz with keys
        for i, v in enumerate(mfcc_means, 1):
            features[f"mfcc_{i}"] = float(v)
        for i, v in enumerate(chroma, 1):
            features[f"chroma_{i}"] = float(v)
        for i, v in enumerate(tonnetz, 1):
            features[f"tonnetz_{i}"] = float(v)
        return features
    except Exception as e:
        print(f"Error extracting {file_path}: {e}")
        return None


In [ ]:
# Parallel feature extraction and caching
def build_feature_dataframe(df_files, cache_file=CACHE_FILE, force_recompute=False, n_workers=6):
    cache_file = Path(cache_file)
    if cache_file.exists() and not force_recompute:
        print("Loading cached features from", cache_file)
        df = load(cache_file)
        return df
    print("Extracting features (this may take a while)...")
    records = []
    files = df_files.to_dict("records")
    with ProcessPoolExecutor(max_workers=n_workers) as exe:
        futures = {exe.submit(extract_features, rec["path"]): rec for rec in files}
        for fut in tqdm(as_completed(futures), total=len(futures)):
            rec = futures[fut]
            feat = fut.result()
            if feat is None:
                continue
            feat["path"] = rec["path"]
            feat["genre"] = rec["genre"]
            records.append(feat)
    df = pd.DataFrame(records)
    print(f"Extracted features for {len(df)} files.")
    # save cache
    dump(df, cache_file)
    print("Saved feature cache to", cache_file)
    return df

# Run extraction (will cache)
df_features = build_feature_dataframe(df_files, cache_file=CACHE_FILE, force_recompute=False, n_workers=6)
df_features.shape


In [ ]:
# Basic clean and overview
print(df_features.columns.tolist()[:20])
df_features = df_features.dropna().reset_index(drop=True)
print("Genres:", df_features['genre'].unique())
print(df_features.groupby('genre').size())
df_features.head()


# Visualization:
- distribution of genres
- tempo distribution by genre
- PCA on features to see separability


In [ ]:
# Genre counts
plt.figure(figsize=(10,4))
sns.countplot(data=df_features, x='genre', order=df_features['genre'].value_counts().index)
plt.xticks(rotation=45)
plt.title("Files per Genre")
plt.tight_layout()
plt.show()

# Tempo distribution by genre
plt.figure(figsize=(10,5))
sns.violinplot(data=df_features, x='genre', y='tempo', order=df_features['genre'].value_counts().index)
plt.xticks(rotation=45)
plt.title("Tempo distribution by Genre")
plt.tight_layout()
plt.showb()


In [ ]:
# Prepare features and labels
feature_cols = [c for c in df_features.columns if c not in ("path","genre")]
X = df_features[feature_cols].values
y = df_features['genre'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Encoded classes:", list(zip(le.classes_, le.transform(le.classes_))))

# train test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=RANDOM_STATE, stratify=y_enc)
print("Train/test sizes:", X_train.shape, X_test.shape)


In [ ]:
# PCA for visualization of separability
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9,6))
palette = sns.color_palette("tab10", n_colors=len(le.classes_))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=le.inverse_transform(y_enc), palette=palette, s=40, alpha=0.8)
plt.title("PCA (2 components) of audio features")
plt.legend(loc='best', bbox_to_anchor=(1.05,1))
plt.tight_layout()
plt.show()

print("Explained variance ratios (2 components):", pca.explained_variance_ratio_)


## Modeling Plan
1. Build a classification pipeline: `StandardScaler` → `RandomForestClassifier`.
2. Evaluate with cross-validation and test set.
3. Optionally grid-search parameters.
4. Also build an SVM pipeline for comparison.
5. Save best model for deployment.


In [ ]:
# Random Forest pipeline
rf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

rf_params = {
    "rf__n_estimators": [100, 200],
    "rf__max_depth": [None, 20],
    "rf__min_samples_split": [2, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid_rf = GridSearchCV(rf_pipe, rf_params, cv=cv, scoring="accuracy", n_jobs=-1, verbose=1)
grid_rf.fit(X_train, y_train)

print("Best RF params:", grid_rf.best_params_)
print("Best RF CV score:", grid_rf.best_score_)


In [ ]:
# Test evaluation and confusion matrix
best_rf = grid_rf.best_estimator_
y_pred = best_rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Test Accuracy (RF):", acc)
print("Classification report:\n", classification_report(y_test, y_pred, target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (Random Forest)")
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# SVM pipeline comparison
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(probability=True, random_state=RANDOM_STATE))
])

svm_params = {
    "svc__C": [0.1, 1, 10],
    "svc__kernel": ["rbf"]
}

grid_svm = GridSearchCV(svm_pipe, svm_params, cv=cv, scoring="accuracy", n_jobs=-1, verbose=1)
grid_svm.fit(X_train, y_train)

print("Best SVM params:", grid_svm.best_params_)
print("Best SVM CV score:", grid_svm.best_score_)

# Evaluate on test
best_svm = grid_svm.best_estimator_
y_pred_svm = best_svm.predict(X_test)
print("Test Accuracy (SVM):", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))


In [ ]:
# Feature importance from RF
rf_model = best_rf.named_steps['rf']
scaler_for_plot = best_rf.named_steps['scaler']
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(30)
plt.figure(figsize=(10,8))
sns.barplot(x=feat_imp.values, y=feat_imp.index)
plt.title("Top 30 feature importances (Random Forest)")
plt.tight_layout()
plt.show()


## Multivariable Linear Regression
The project request asked explicitly to implement a multivariable linear regression model on a large/complex dataset.  
We will train a linear regression model to predict **mean spectral centroid** (a continuous audio feature related to "brightness") using the same set of features (excluding `spec_centroid` as target).
This helps analyze relationships between features and a continuous audio descriptor, and to inspect residuals (implications for users / business).


In [ ]:
# Prepare data for regression: predict spec_centroid from other features
target = "spec_centroid"
reg_features = [c for c in feature_cols if c != target]

X_reg = df_features[reg_features].values
y_reg = df_features[target].values

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE)

reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])

reg_pipeline.fit(Xr_train, yr_train)
yr_pred = reg_pipeline.predict(Xr_test)

# Evaluate regression
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(yr_test, yr_pred)
r2 = r2_score(yr_test, yr_pred)
print(f"Regression MSE: {mse:.4f}, R^2: {r2:.4f}")

# Save regression model
dump(reg_pipeline, REG_MODEL_FILE)
print("Saved regression model to", REG_MODEL_FILE)


In [ ]:
# Residuals plot
residuals = yr_test - yr_pred
plt.figure(figsize=(8,5))
plt.scatter(yr_pred, residuals, alpha=0.6)
plt.axhline(0, color='k', linestyle='--')
plt.xlabel("Predicted spec_centroid")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted (Linear Regression)")
plt.tight_layout()
plt.show()

# Show a few predicted vs true values
pd.DataFrame({"y_true": yr_test[:10], "y_pred": yr_pred[:10], "residual": residuals[:10]})


In [ ]:
# Save best classifier
dump({
    "model": best_rf,
    "label_encoder": le,
    "feature_columns": feature_cols,
    "scaler": None
}, MODEL_FILE)
print("Saved classifier to", MODEL_FILE)


In [ ]:
# Write a minimal train script for DevOps to reproduce training & save model
train_script = r'''
#!/usr/bin/env python3
"""
train.py
Minimal script to train and save the RandomForest classifier using precomputed features cache.
Usage:
    python train.py --cache features_cache.joblib --out model.joblib
"""
import argparse
from joblib import load, dump
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import numpy as np

def main(cache, out, random_state=42):
    data = load(cache)
    feature_cols = [c for c in data.columns if c not in ("path","genre")]
    X = data[feature_cols].values
    y = data["genre"].values
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=random_state)
    pipeline = Pipeline([("scaler", StandardScaler()), ("rf", RandomForestClassifier(random_state=random_state,n_jobs=-1))])
    params = {"rf__n_estimators":[100], "rf__max_depth":[None], "rf__min_samples_split":[2]}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    grid = GridSearchCV(pipeline, params, cv=cv, scoring="accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    model = {"model": grid.best_estimator_, "label_encoder": le, "feature_columns": feature_cols}
    dump(model, out)
    print("Saved model to", out)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--cache", required=True)
    parser.add_argument("--out", default="genre_classifier.joblib")
    args = parser.parse_args()
    main(args.cache, args.out)
'''
with open("train.py", "w") as f:
    f.write(train_script)
os.chmod("train.py", 0o755)
print("Wrote train.py")


In [ ]:
# Create a minimal PowerPoint summary using python-pptx
from pptx import Presentation
from pptx.util import Inches, Pt

prs = Presentation()
title_slide_layout = prs.slide_layouts[0]
slide = prs.slides.add_slide(title_slide_layout)
slide.shapes.title.text = "Classically Punk — Genre Classification"
slide.placeholders[1].text = "Dataset: Tzanetakis et al. | Features: MFCC, Chroma, Spectral features\nModels: Random Forest, SVM, Linear Regression"

# Add a slide about results
sl_layout = prs.slide_layouts[1]
slide = prs.slides.add_slide(sl_layout)
slide.shapes.title.text = "Model Results"
body = slide.shapes.placeholders[1].text_frame
body.text = f"Random Forest test accuracy: {accuracy_score(y_test, y_pred):.3f}"
p = body.add_paragraph()
p.text = f"SVM test accuracy: {accuracy_score(y_test, y_pred_svm):.3f}"
p.level = 1
p = body.add_paragraph()
p.text = f"Regression R^2 predicting spectral centroid: {r2:.3f}"
p.level = 1

prs.save(PPTX_FILE)
print("Saved presentation to", PPTX_FILE)


End !!!